# Python and CLI for Data Science - Session 12

- *Course*: Python and CLI for Data Science
- *Session*: 12
- *Unit*: SciPy Hypothesis Testing

## SciPy Hypothesis Testing

### (Re)sources:
- [Statistical Hypothesis Tests in Python - Cheat Sheet](https://machinelearningmastery.com/statistical-hypothesis-tests-in-python-cheat-sheet/) _by Jason Brownlee_


- In general, hypothesis testing works by computing a test statistic from the data and comparing it to a critical value
- If the test statistic is greater than the critical value, we reject the null hypothesis; otherwise we fail to reject it

- Different statistical tests exist, depending on
    - the type of data
    - the question we want to answer
    - the assumptions the test makes about the data
- We will take a look at how to do statistical testing with Python in this session

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ipywidgets import interact
from scipy import stats
import requests
import io

def download(url):
    return pd.read_csv(io.StringIO(requests.get(url).content.decode("utf-8")))

# %matplotlib notebook
%matplotlib ipympl

## Animated Plots

- In this session, we will make heavy use of animated plots to visualize statistical tests
- We will use the `interact` decorator to update values inside a plot
- The following example illustrates how this works

In [ ]:
# if this example does not work for you, try running
# `jupyter nbextension install --user --py widgetsnbextension`
# `jupyter nbextension enable --user --py widgetsnbextension`

fig, ax = plt.subplots()


@interact(x=(0, 5, 0.1), y=(0, 5, 0.1))
def update(x, y):
    ax.clear()
    ax.set(xlim=(0, 5), ylim=(0, 5))
    ax.scatter(x, y, marker="x")

## SciPy

- SciPy is a python library for scientific computing
- We will only take a look at the SciPy stats module in this session
- SciPy stats contains a large number of statistical functions, including probability distributions, statistical functions, and statistical tests

- For example, we can use the `norm.pdf` function to generate a normal distribution with a given mean and variance
- `pdf` stands for the probability density function
- When integrating over the `pdf` we obtain the probability that is less than or equal to that value
- As such, the integral from -inf to inf of a probability density function is always 1

In [ ]:
x = np.linspace(-5, 5, 100)
y = stats.norm.pdf(x, 0, 1)

fig, ax = plt.subplots()

@interact(val=(-5, 5, 0.1))
def update(val):
    ax.clear()
    ax.plot(x, y)
    pdf_val = stats.norm.pdf(val, 0, 1)
    ax.text(0.8, 0.9, f"{pdf_val:.2f}", transform=ax.transAxes)
    ax.plot(val, pdf_val, "ro")

- SciPy gives us direct access to the integral of `pdf` as the `cdf` (cumulative density function) 
- The `cdf` returns the probability of a value less than or equal to a given x value

In [ ]:
fig, ax1 = plt.subplots()

pdf = stats.norm.pdf(x, 0, 1)
cdf = stats.norm.cdf(x, 0, 1)
ax1.tick_params("y", colors="blue")
ax2 = ax1.twinx()
ax2.plot(x, cdf, color="red")
ax2.tick_params("y", colors="red")
scatter = ax2.scatter([0], [stats.norm.cdf(-5, 0, 1)], marker="x", color="black")


@interact(x_fill=(-5, 5, 0.1))
def update(x_fill=0):
    ax1.clear()
    ax1.plot(x, pdf, c="blue")
    ax1.fill_between(x, pdf, where=(x <= x_fill), color="blue", alpha=0.3)
    scatter.set_offsets([[x_fill, stats.norm.cdf(x_fill, 0, 1)]])
    ax1.text(0.1, 0.8, f"{stats.norm.cdf(x_fill, 0, 1) * 100:0>6.2f}", transform=ax.transAxes)

- The `pdf` and `cdf` of various probability distributions are the underlying concepts with which statistical tests work
- The test considers a test statistic which is distributed according to a probability distribution
- If the value of the test statistic is greater than a critical value than we can reject the null hypothesis and speak of a statistically significant result

- For example, consider the following distribution which models the height of humans in the population (mu = 180 cm, sigma = 20 cm)
- If we set a critical value of 5%, which people are significantly taller/shorter than the average human?

In [ ]:
fig, ax1 = plt.subplots()
x = np.linspace(80, 280, 100)
pdf = stats.norm.pdf(x, 180, 20)
cdf = stats.norm.cdf(x, 180, 20)
ax1.plot(x, pdf)
ax1.tick_params("y", colors="blue")
ax2 = ax1.twinx()
ax2.plot(x, cdf, color="red")
ax2.plot(x, 1 - cdf, color="red")
ax2.tick_params("y", colors="red")
scatter = ax2.scatter([80], [stats.norm.cdf(80, 180, 20)], marker="x", color="black")


@interact(x_fill=(80, 280, 0.1))
def update(x_fill=0):
    ax1.clear()
    ax1.plot(x, pdf, color="blue")
    ax1.fill_between(x, pdf, where=(x <= x_fill), color="blue", alpha=0.3)
    scatter.set_offsets([[x_fill, stats.norm.cdf(x_fill, 180, 20)]])
    cdf_1 = stats.norm.cdf(x_fill, 180, 20)
    cdf_2 = 1 - stats.norm.cdf(x_fill, 180, 20)
    ax1.text(0.8, 0.8, f"{cdf_1 * 100:0>6.2f}%\n{cdf_2 * 100:0>6.2f}%", transform=ax.transAxes)

- Statistical tests are not restricted to normal distributions
- In fact, most do not use a normal distribution for the test statistic
- In this session we will not take a deeper look into the inner workings of most statistical tests, but keep this fact in mind

- One final important concept for statistical tests is the sample size
- The more random samples we draw from a distribution, the closer our sample mimics the actual true distribution
- We can visualize this by sampling random values using the `rvs` function with different sample sizes

In [ ]:
fig, ax = plt.subplots()
x = np.linspace(-5, 5, 100)
y = stats.norm.pdf(x, 0, 1)
ax.plot(x, y)
xlim = ax.get_xlim()
ylim = ax.get_ylim()


@interact(samples=(1, 5000, 10))
def update(samples):
    ax.clear()
    ax.plot(x, y)
    sample = stats.norm.rvs(0, 1, size=samples)
    ax.hist(sample, density=True, bins=int(np.sqrt(samples)), alpha=0.5)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

- In consequence, the higher the sample size the more robust and informative a statistical test becomes

## Non-parametric Tests

### Permutation Test

- Tests whether an arbitrary given statistic is significantly different between two samples
- The test combines the data from both samples
- It then iteratively repeats the following steps
    - Randomly shuffle the data
    - Split the data into two samples of the original size
    - Compute and store the statistic

- This is done multiple times to determine how (un)likely the original sample's test statistic is

---

- Assumptions:
    - Observations in each sample are independent and identically distributed (iid)
  
- Interpretation:
    - H0: The Data is randomly sampled from the same distribution
    - H1: The Data is not randomly sampled from the same distribution

- Let's take two samples from two different normal distributions and determine whether their means are significantly different from one another using `stats.permutation_test`

In [ ]:
rvs1 = stats.norm.rvs(1, size=10)
rvs2 = stats.norm.rvs(2, size=10)

def statistic(x, y):
    return np.mean(x) - np.mean(y)

permutation_result = stats.permutation_test((rvs1, rvs2), statistic)
permutation_result

- How do we interpret this result?
    1. The statistic is the actual difference measured between the two samples
    2. The pvalue tells us if our result is significant or not. Is the value smaller than our critical value (e.g. 0.05)?
    3. The null distribution is an array containing the computed statistics from the random samples drawn from the joint sample

In [ ]:
print(permutation_result.statistic)
print(permutation_result.pvalue)
print(permutation_result.null_distribution, permutation_result.null_distribution.shape)

- Let's do the test manually to understand what is happening

In [ ]:
actual_statistic = statistic(rvs1, rvs2)
statistics = []
rvs = np.concatenate([rvs1, rvs2])
for _ in range(1000):
    np.random.shuffle(rvs)
    _rvs1, _rvs2 = np.array_split(rvs, 2)
    statistics.append(statistic(_rvs1, _rvs2))
random_statistics = np.array(statistics)
random_statistics.shape

- Now we have random samples of our statistic
- How do we determine if our actual difference is significant?
- We compare our actual statistic to the distribution of the random statistics by computing which percentile of the random statistics is smaller/greater than our actual statistic

In [ ]:
smaller = (random_statistics <= actual_statistic).sum()
pvalue = percentage_smaller = smaller / random_statistics.shape[0]
pvalue

- Let's visualize what we've computed to get a better understanding
- We plot the histogram of the random statistics along with our actual computed statistic

In [ ]:
fig, ax = plt.subplots()

ax.hist(random_statistics, density=True, bins=int(random_statistics.shape[0] ** 0.5))
ax.axvline(actual_statistic, color="red")

- Let's investigate how the number of samples and the difference between means affects the permutation test

In [ ]:
fig, ax = plt.subplots()


@interact(n=(10, 100, 5), mu1=(1, 10, 0.1), mu2=(1, 10, 0.1))
def update(n, mu1, mu2):
    x = stats.norm.rvs(size=n, loc=mu1)
    y = stats.norm.rvs(size=n, loc=mu2)

    result = stats.permutation_test((x, y), statistic)
    ax.clear()
    ax.set(xlim=(-1.5, 1.5), ylim=(0, 2.5))
    ax.hist(result.null_distribution, bins=int(result.null_distribution.size**0.5), density=True)
    ax.axvline(result.statistic, color="red")
    ax.text(0.05, 0.95, f"{result.pvalue:.4f}", transform=ax.transAxes)

**Note**

- The permutation test is very flexible
- We can compute any statistic we want to
- For example, we could compute the variance or standard deviation instead of the mean
- This flexibility makes the test applicable in almost any scenario, but makes it harder to find significant results

### Exercise

- Determine whether the means of a feature (e.g., the sepal length) between different flower species from the iris dataset are significantly different from one another

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["species"] = iris.target
iris_df["species"] = iris_df["species"].map({idx: target_name for idx, target_name in enumerate(iris.target_names)})
iris_df

In [ ]:
# feature options are: "sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"
feature = "sepal width (cm)"
# flower options are: "setosa", "versicolor", "virginica"
flower1 = "setosa"
flower2 = "versicolor"

feature1 = iris_df.loc[iris_df["species"] == flower1, feature]
feature2 = iris_df.loc[iris_df["species"] == flower2, feature]

# compute the permutation test for the difference of mean values for different features and flower species
# which features for which pairs of flowers have a significantly different mean value?

In [ ]:
permutation_result = stats.permutation_test((feature1, feature2), statistic)
permutation_result

## Parametric Hypothesis Tests

### Student's t-test

- Determining whether the means between two samples are significantly different from one another is a very common case and can be measured with the Student's t-test
- Student's t-test is one of the most commonly applied tests
- It uses a different underlying distribution as the permutation test and makes more strict assumptions
- It can, therefore, also detect more nuanced differences than the permutation test, but has stricter requirements

---

- Assumptions:
    - Observations in each sample are independent and identically distributed (iid)
    - Observations in each sample are normally distributed
    - Observations in each sample have the same variance

- Interpretation:
    - H0: the means of the samples are equal
    - H1: the means of the samples are unequal

- Let's compare two samples from normal distributions and check whether their mean values are different using `stats.ttest_ind`

In [ ]:
rvs1 = stats.norm.rvs(1, size=100)
rvs2 = stats.norm.rvs(2, size=100)

ttest_result = stats.ttest_ind(rvs1, rvs2)
ttest_result

- In the result we again have three values
    1. statistic (the computed test statistic; **not the difference between means for the student t-test**)
    2. pvalue (probability of the statistic being random)
    3. df (degrees of freedom)

In [ ]:
print(ttest_result.statistic)
print(ttest_result.pvalue)
print(ttest_result.df)

- Let's investigate how the sample size and difference between means affects the t-test

In [ ]:
fig, ax = plt.subplots()

@interact(n=(10, 100, 5), mu1=(1, 10, 0.1), mu2=(1, 10, 0.1))
def update(n, mu1, mu2):
    rvs_list = []
    ax.clear()
    colors = ["blue", "red"]
    for idx, mu in enumerate((mu1, mu2)):
        rvs = stats.norm.rvs(mu, size=n)
        rvs_list.append(rvs)
        x = np.linspace(rvs.min(), rvs.max(), 100)
        pdf = stats.norm.pdf(x, rvs.mean(), rvs.var())
        ax.hist(rvs, bins=int(n**0.5), alpha=0.5, density=True, color=colors[idx])
        ax.plot(x, pdf, "--", color=colors[idx])
    ax.set(ylim=(0, 0.5))
    ttest_result = stats.ttest_ind(rvs_list[0], rvs_list[1])
    sig = ttest_result.pvalue < 0.05
    sig_text = "Significantly different" if sig else "Not significantly different"
    ax.text(0.05, 0.95, f"{ttest_result.pvalue:.3f}, {sig_text}", transform=ax.transAxes)

### Exercise

- Determine whether the means of a feature between different flower species from the iris dataset are significantly different from one another

- This time, use the Student's t-test

In [ ]:
# feature options are: "sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"
feature = "petal length (cm)"
# flower options are: "setosa", "versicolor", "virginica"
flower1 = "setosa"
flower2 = "versicolor"

feature1 = iris_df.loc[iris_df["species"] == flower1, feature]
feature2 = iris_df.loc[iris_df["species"] == flower2, feature]

# compute the student's t-test for the different features and flower species
# which features for which pairs of flowers have a significantly different mean value?

In [ ]:
ttest_result = stats.ttest_ind(feature1, feature2)
ttest_result

## Testing for the same distribution

- Often we want to test if two samples are drawn from the same distribution
- We will take a look at two tests
    - Kolmogorov-Smirnov Test
    - Shapiro-Wilk Normality Test

### Kolmogorov-Smirnov Test

- To test if two samples come from the same distribution, we can use the Kolmogorov-Smirnov test
- Since the test makes virtually no assumptions, it is very general and can be applied in most cases

---

- Assumptions:
  - Observations in each sample are independent and identically distributed (iid)

- Interpretation:
  - H0: the two samples are drawn from the same distribution
  - H1: the two samples are drawn from different distributions

- Let's compare 2 samples drawn from 2 different normal distributions using `stats.ks_2samp`
- The main output of interest is again the `pvalue`

In [ ]:
mu1 = 0
mu2 = 0
sigma1 = 1
sigma2 = 20 # same mu different sigma
size = 100

rvs1 = stats.norm.rvs(mu1, sigma1, size=size)
rvs2 = stats.norm.rvs(mu2, sigma2, size=size)

ks_result = stats.ks_2samp(rvs1, rvs2)
ks_result

- We can also compare different distributions with one another

In [ ]:
fig, ax = plt.subplots()

@interact(
    n=(10, 100, 5),
    mu1=(1, 10, 0.1),
    mu2=(1, 10, 0.1),
    dist1=["norm", "laplace", "chi2"],
    dist2=["norm", "laplace", "chi2"],
)
def update(n, mu1, mu2, dist1, dist2):
    rvs_list = []
    ax.clear()
    colors = ["blue", "red"]
    for idx, (mu, dist) in enumerate(zip((mu1, mu2), (dist1, dist2))):
        dist = getattr(stats, dist)  # identical to stats.norm or stats.poisson
        rvs = dist.rvs(mu, size=n)
        rvs_list.append(rvs)
        x = np.linspace(rvs.min(), rvs.max(), 100)
        pdf = dist.pdf(x, mu)
        ax.hist(rvs, bins=int(n**0.5), alpha=0.5, label=f"dist{idx}", density=True, color=colors[idx])
        ax.plot(x, pdf, "--", color=colors[idx])
    ax.plot()
    ax.legend()
    ks_result = stats.ks_2samp(rvs_list[0], rvs_list[1])
    sig = ks_result.pvalue < 0.05
    sig_text = "Significantly different" if sig else "Not significantly different"
    ax.text(0.05, 0.95, f"{ks_result.pvalue:.3f}, {sig_text}", transform=ax.transAxes)

### Exercise

- Compare the distributions of various features for the English Premier League for home and away teams

In [ ]:
url = (
    "https://raw.githubusercontent.com/tara-nguyen/english-premier-league-datasets-"
    "for-10-seasons/main/epldat10seasons/epl-allseasons-matchstats.csv"
)
pl_df = download(url)
pl_df

In [ ]:
# available features: "Goals", "Shots", "Corners", "YellowCards", "RedCards"
feature = "Goals"

fig, ax = plt.subplots()

home = pl_df[f"Home{feature}"]
away = pl_df[f"Away{feature}"]

ax.hist(home, alpha=0.5, label="Home", bins=home.max())
ax.hist(away, alpha=0.5, label="Away", bins=away.max())
ax.legend()
ax.set_title(feature)

# compute the ks test statistic
# then add the p value and whether the distributions are significantly different to one another to the plot

In [ ]:
ks_result = stats.ks_2samp(home, away)
sig = ks_result.pvalue < 0.05
sig_text = "Significantly different" if sig else "Not significantly different"
ax.text(0.05, 0.95, f"{ks_result.pvalue:.3f}, {sig_text}", transform=ax.transAxes)

### Shapiro-Wilk Normality Test

- A special case for testing for a distribution is the Shapiro-Wilk test for normality
- It tests whether a sample comes from a normal distribution
- We could also test this with the Kolmogorov-Smirnov test by comparing a sample with samples drawn from a normal distribution with the same mean and variance
- Because the Shapiro-Wilk test only tests for normality, it makes stronger assumptions and can therefore identify smaller differences between distributions as significant

---

- Assumptions:
  - Observations in each sample are independent and identically distributed (iid)

- Interpretation:
  - H0: the sample has a Gaussian distribution
  - H1: the sample does not have a Gaussian distribution

- Let's test several different distributions to see if the Shapiro-Wilk test identifies them as normal

In [ ]:
mu = 1
sigma = 1
size = 100

norm_rvs = stats.norm.rvs(mu, sigma, size=size)
laplace_rvs = stats.laplace.rvs(mu, sigma, size=size)
chi2_rvs = stats.chi2.rvs(mu, sigma, size=size)

norm_shapiro_result = stats.shapiro(norm_rvs)
laplace_shapiro_result = stats.shapiro(laplace_rvs)
chi2_shapiro_result = stats.shapiro(chi2_rvs)
norm_shapiro_result, laplace_shapiro_result, chi2_shapiro_result

- The Shapiro-Wilk test is able to better identify that the Laplace distribution is not a normal distribution compared to the Kolmogorov-Smirnov test

- Let's visualize the test and see how the number of samples influence the test

In [ ]:
fig, ax = plt.subplots()


@interact(n=(10, 100, 5), sigma=(1, 10, 0.1), dist=["norm", "laplace", "chi2"])
def update(n, sigma, dist):
    dist = getattr(stats, dist)  # identical to stats.norm or stats.poisson
    ax.clear()
    rvs = dist.rvs(0, sigma, size=n)
    x = np.linspace(rvs.min(), rvs.max(), 100)
    ax.hist(rvs, bins=int(n**0.5), alpha=0.5, density=True, color="blue")
    pdf = dist.pdf(x, rvs.mean(), sigma)
    ax.plot(x, pdf, "--", color="blue")

    x = np.linspace(-sigma * 5, sigma * 5, 100)
    pdf = stats.norm.pdf(x, 0, sigma)
    ax.plot(x, pdf, "--", color="red")

    shapiro_result = stats.shapiro(rvs)
    sig = shapiro_result.pvalue < 0.05
    sig_text = "Significantly different" if sig else "Not significantly different"
    ax.text(0.05, 0.95, f"{shapiro_result.pvalue:.3f}, {sig_text}", transform=ax.transAxes)

### Exercise

- Test if the distribution of various metrics from the English Premier League is (not) normal

In [ ]:
# available features: "Goals", "Shots", "Corners", "YellowCards"
feature = "Goals"

fig, ax = plt.subplots()

ser = pl_df[f"Home{feature}"] + pl_df[f"Away{feature}"]

x = np.linspace(ser.min(), ser.max(), 100)

ax.hist(ser, density=True)
ax.plot(x, stats.norm.pdf(x, ser.mean(), ser.std()), "--")
ax.set_title(feature)

# compute the shapiro-wilk test statistic
# then add the p value and whether the distribution significantly different from a normal distribution to the plot

In [ ]:
shapiro_result = stats.shapiro(ser)
sig = shapiro_result.pvalue < 0.05
sig_text = "Significantly different" if sig else "Not significantly different"
ax.text(0.05, 0.95, f"{shapiro_result.pvalue:.3f}, {sig_text}", transform=ax.transAxes)

## Correlation Tests

- Correlation tests measure whether a change in one variable coincides with a change in another variable (and vice versa)
- Importantly, correlation does not measure whether the variables influence each other and in which way
- It only measures if the variables change in a similar matter

- We will take a look at two correlation tests:
    - Pearson's/Spearman's Correlation Coefficient
    - Chi-Squared Test

### Pearson's/Spearman's Correlation Coefficient

- Both tests determine whether two samples have a relationship
- The difference betweent the two is that
    - Pearson's correlation coefficient considers the differences between the values of two samples
    - Spearman's correlation coefficient considers if a value is smaller/greater than another value

- Spearman's correlation coefficient can, therefore, be used to ordinal samples where differences between values are not meaningful

---

- Assumptions:
    - Observations in each sample are independent and identically distributed (iid)
    - Observations in each sample are normally distributed
    - Observations in each sample have the same variance

- Interpretation
    - H0: the two samples are independent
    - H1: there is a dependency between the samples

- Let's measure the correlation of a linear function with some noise added to it

In [ ]:
fig, ax = plt.subplots()

x = stats.uniform.rvs(-2, 4, size=50)
noise = stats.norm.rvs(0, 1, size=50)
y = 2 * x + noise

ax.scatter(x, y)
pearson_result = stats.pearsonr(x, y)
pearson_result

- Let's visualize the correlation for a few different functions

In [ ]:
fig, ax = plt.subplots()

funcs = {
    "y=a*x": lambda x: x,
    "y=a*x^2": lambda x: x**2,
    "y=a*exp(x)": lambda x: np.exp(x),
}


@interact(
    n=(10, 100, 10),
    a=(-2, 2, 0.1),
    w=(0, 1.5, 0.01),
    func=funcs.keys(),
    corr_func=["pearson", "spearman"],
)
def update(n, a, w, func, corr_func):
    x = stats.uniform.rvs(-2, 4, size=n)
    y = funcs[func](x) * a + stats.norm.rvs(0, w, size=n)
    ax.clear()
    ax.set(xlim=(-2.2, 2.2), ylim=(-2.2, 2.2))
    ax.scatter(x, y)
    if corr_func == "pearson":
        corr_coeff, p_value = stats.pearsonr(x, y)
    elif corr_func == "spearman":
        corr_coeff, p_value = stats.spearmanr(x, y)
    else:
        raise ValueError("corr_func must be either pearson or spearman")
    sig_text = "Signficant correlation" if p_value < 0.05 else "No significant correlation"
    ax.text(0.05, 0.95, f"{corr_coeff:.2f}, {sig_text}", transform=ax.transAxes)

### Exercise

- Measure the correlation of various features with the disease progression value for diabetes

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
diabetes_df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
diabetes_df["disease_progression"] = diabetes.target
diabetes_df

In [ ]:
# Options: age, sex, bmi, bp, s1, s2, s3, s4, s5, s6
feature = "bp"

x = diabetes_df[feature]
y = diabetes_df["disease_progression"]

fig, ax = plt.subplots()

ax.scatter(x, y)

# compute the pearson or spearman rank correlation
# then add the correlation coefficient and whether the correlation is significant to the plot

In [ ]:
corr_coeff, p_value = stats.pearsonr(x, y)
sig_text = "Signficant correlation" if p_value < 0.05 else "No significant correlation"
ax.text(0.05, 0.95, f"{corr_coeff:.2f}, {sig_text}", transform=ax.transAxes)

### Chi-Squared Test

- Tests whether two categorical variables are related or independent
- Often we cannot order our values, but simply have frequencies for different categories
- To measure if two categorical features correlate with one another we can use the Chi-Squared test

---

- Assumptions:
    - Observations used in the calculation of the contingency table are independent
    - 5 or more examples in each cell of the contingency table

- Interpretation:
    - H0: the two samples are independent
    - H1: there is a dependency between the samples

- For example, say we had one variable with categories A and B, and another with values C and D
- In our sample we found the following frequencies for the different categories

|        |   A |   B |
|:-------|----:|----:|
| **C**  |  24 |  22 |
| **D**  |  15 |  39 |

- Let's measure if the two categories correlate with one another

In [ ]:
df = pd.DataFrame([[24, 22], [15, 39]], index=["C", "D"], columns=["A", "B"])
chi2_result = stats.chi2_contingency(df)
chi2_result

- Let's investigate how the sample size and bias affect the Chi-Squared test

In [ ]:
def update(n, p):
    x = stats.multinomial.rvs(n, p=[(1 - p) / 2, p / 2, p / 2, (1 - p) / 2])
    x = x.reshape(2, 2)
    df = pd.DataFrame(x, columns=["A", "B"], index=["C", "D"])
    display(df)
    _, p, _, expected = stats.chi2_contingency(df)
    sig_text = "Signficant correlation" if p < 0.05 else "No significant correlation"
    expected_df = pd.DataFrame(expected, columns=["A", "B"], index=["C", "D"])
    display(expected_df)
    display(f"{p:.4f}, {sig_text}")


interact(update, n=(10, 1000, 10), p=(0, 1, 0.01));

### Exercise

- Determine if there is a significant correlation between the variables sex or class and whether a person survived the sinking of the Titanic

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
titanic_df = download(url)
titanic_df

In [ ]:
variable = "class"  # or sex

table = titanic_df.loc[:, ["survived", variable]].value_counts().unstack()
display(table)

# compute the chi-squared test for contingency tables
# then print the expected counts if there is not correlation and
# whether the correlation between the two variables is significant

_, p, _, expected = stats.chi2_contingency(table)
print(p)
display(pd.DataFrame(expected, index=table.index, columns=table.columns))

In [ ]:
_, p, _, expected = stats.chi2_contingency(table)
print(p)
display(pd.DataFrame(expected, index=table.index, columns=table.columns))